# Python Software Engineering for ML

ML engineers write production code, not just notebooks. This means type hints, dataclasses, testing, logging, and clean APIs that withstand real traffic.

**Topics:** Type hints, dataclasses, context managers, testing with pytest, logging, CLI with argparse

In [ ]:
# Type hints: make code self-documenting and enable IDE support
from typing import Optional, List, Dict, Tuple, Union, Callable, Any
from typing import TypeVar, Generic
import time
import logging
import functools

# Basic type hints
def preprocess(
    texts: List[str],
    max_length: int = 512,
    lowercase: bool = True,
    stopwords: Optional[List[str]] = None,
) -> List[str]:
    """Preprocess a list of texts."""
    stopwords = stopwords or []
    result = []
    for text in texts:
        if lowercase:
            text = text.lower()
        tokens = [t for t in text.split() if t not in stopwords]
        result.append(' '.join(tokens[:max_length]))
    return result

# Complex type hints with TypedDict
from typing import TypedDict

class ModelConfig(TypedDict):
    model_name: str
    learning_rate: float
    batch_size: int
    max_epochs: int
    early_stopping: bool

class PredictionResult(TypedDict):
    label: str
    confidence: float
    processing_time_ms: float

def predict(text: str, model_config: ModelConfig) -> PredictionResult:
    """Mock prediction function with typed return."""
    start = time.perf_counter()
    # ... model inference ...
    elapsed_ms = (time.perf_counter() - start) * 1000
    return PredictionResult(label='positive', confidence=0.92, processing_time_ms=elapsed_ms)

config: ModelConfig = {
    'model_name': 'distilbert-sentiment',
    'learning_rate': 2e-5,
    'batch_size': 32,
    'max_epochs': 10,
    'early_stopping': True,
}

texts = ["This is great!", "Terrible product"]
processed = preprocess(texts)
print('Type hints in action:')
print(f'  preprocess({texts}) → {processed}')
print(f'  config type check: model_name={config["model_name"]}')

## 2. Dataclasses — Structured Configuration

In [ ]:
from dataclasses import dataclass, field, asdict
import json

@dataclass
class TrainingConfig:
    """Immutable training configuration."""
    model_name: str
    learning_rate: float = 1e-4
    batch_size: int = 32
    max_epochs: int = 50
    warmup_steps: int = 500
    weight_decay: float = 0.01
    gradient_clip: float = 1.0
    early_stopping_patience: int = 5
    feature_columns: List[str] = field(default_factory=list)  # mutable default!
    
    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2)
    
    @classmethod
    def from_json(cls, json_str: str) -> 'TrainingConfig':
        return cls(**json.loads(json_str))
    
    def with_lr(self, lr: float) -> 'TrainingConfig':
        """Return new config with different learning rate (immutable pattern)."""
        from dataclasses import replace
        return replace(self, learning_rate=lr)

@dataclass
class ModelMetrics:
    """Training metrics for a single epoch."""
    epoch: int
    train_loss: float
    val_loss: float
    val_accuracy: float
    learning_rate: float
    epoch_time_s: float
    
    @property
    def is_best(self) -> bool:
        return self.val_accuracy > 0.95  # would compare to historical in practice
    
    def __str__(self) -> str:
        return (f'Epoch {self.epoch:3d} | '
                f'train_loss={self.train_loss:.4f} | '
                f'val_loss={self.val_loss:.4f} | '
                f'val_acc={self.val_accuracy:.4f} | '
                f'lr={self.learning_rate:.2e} | '
                f't={self.epoch_time_s:.1f}s')

config = TrainingConfig(
    model_name='xgboost-v2',
    learning_rate=0.1,
    feature_columns=['age', 'income', 'credit_score']
)

print('Dataclass config:')
print(config)
print()
print('JSON serialization:')
print(config.to_json())

# Immutable update
config_v2 = config.with_lr(0.01)
print(f'\nOriginal lr: {config.learning_rate}')
print(f'New config lr: {config_v2.learning_rate}')

# Metrics
m = ModelMetrics(1, 0.45, 0.38, 0.91, 1e-3, 12.5)
print(f'\n{m}')

## 3. Context Managers & Resource Management

In [ ]:
import contextlib
import time

@contextlib.contextmanager
def timer(name: str = ''):
    """Context manager for timing code blocks."""
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f'[timer] {name}: {elapsed*1000:.2f}ms')

@contextlib.contextmanager
def model_inference_context(model_path: str, device: str = 'cpu'):
    """Load model, run inference, cleanup. Guarantees cleanup even on error."""
    print(f'  → Loading model from {model_path} on {device}')
    model = {'weights': 'loaded', 'device': device}  # mock model
    try:
        yield model
    finally:
        print(f'  → Cleaning up model resources')
        del model

@contextlib.contextmanager
def suppress_warnings():
    """Suppress all warnings in block."""
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        yield

# Usage
with timer('Feature engineering'):
    import numpy as np
    X = np.random.randn(10000, 100)
    X_feat = X @ X.T

with model_inference_context('models/sentiment_v2.pkl') as model:
    print(f'  → Running inference with model: {model}')

# Class-based context manager
class DatabaseConnection:
    def __init__(self, host: str, db: str):
        self.host = host
        self.db = db
        self.connection = None
    
    def __enter__(self):
        print(f'Connecting to {self.host}/{self.db}')
        self.connection = {'host': self.host, 'status': 'connected'}
        return self.connection
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f'Closing connection to {self.host}/{self.db}')
        self.connection = None
        return False  # don't suppress exceptions

print()
with DatabaseConnection('prod-db.company.com', 'ml_features') as conn:
    print(f'Connection status: {conn["status"]}')

## 4. Testing with pytest

In [ ]:
# In production: write test files (test_preprocessing.py, test_model.py)
# Here: demonstrate the patterns inline

# The functions to test
def compute_feature_stats(data: List[float]) -> Dict[str, float]:
    """Compute basic statistics for a feature."""
    if not data:
        raise ValueError('Cannot compute stats on empty data')
    arr = np.array(data)
    return {
        'mean': float(arr.mean()),
        'std': float(arr.std()),
        'min': float(arr.min()),
        'max': float(arr.max()),
        'median': float(np.median(arr)),
    }

def normalize(data: List[float], mean: float, std: float) -> List[float]:
    """Z-score normalize data using provided statistics."""
    if std == 0:
        return [0.0] * len(data)
    return [(x - mean) / std for x in data]

# Pytest-style test cases (run with: pytest test_features.py -v)
import pytest

class TestComputeFeatureStats:
    def test_basic_stats(self):
        data = [1.0, 2.0, 3.0, 4.0, 5.0]
        stats = compute_feature_stats(data)
        assert stats['mean'] == 3.0
        assert stats['min'] == 1.0
        assert stats['max'] == 5.0
    
    def test_empty_input_raises(self):
        with pytest.raises(ValueError, match='empty'):
            compute_feature_stats([])
    
    def test_single_element(self):
        stats = compute_feature_stats([42.0])
        assert stats['mean'] == 42.0
        assert stats['std'] == 0.0

class TestNormalize:
    def test_standard_normalization(self):
        data = [1.0, 2.0, 3.0]
        result = normalize(data, mean=2.0, std=1.0)
        assert result == [-1.0, 0.0, 1.0]
    
    def test_zero_std_returns_zeros(self):
        result = normalize([5.0, 5.0, 5.0], mean=5.0, std=0.0)
        assert all(x == 0.0 for x in result)
    
    def test_preserves_length(self):
        data = list(range(100))
        result = normalize(data, mean=50.0, std=29.0)
        assert len(result) == len(data)

# Run tests manually (in Jupyter)
import traceback

def run_tests(test_class):
    obj = test_class()
    passed = failed = 0
    for name in dir(obj):
        if name.startswith('test_'):
            try:
                getattr(obj, name)()
                print(f'  ✓ {name}')
                passed += 1
            except Exception as e:
                print(f'  ✗ {name}: {e}')
                failed += 1
    return passed, failed

print('Running tests:')
for cls in [TestComputeFeatureStats, TestNormalize]:
    print(f'\n{cls.__name__}:')
    p, f = run_tests(cls)
    print(f'  → {p} passed, {f} failed')

## 5. Logging — Production-Grade

In [ ]:
import logging
import sys
from datetime import datetime

def setup_logger(name: str, level: int = logging.INFO) -> logging.Logger:
    """Create a production-ready logger with structured output."""
    logger = logging.getLogger(name)
    logger.setLevel(level)
    
    if logger.handlers:
        return logger  # avoid adding duplicate handlers
    
    # Console handler with structured format
    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter(
        fmt='%(asctime)s | %(name)-20s | %(levelname)-8s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    return logger

# Logging decorator for ML pipeline steps
def log_step(logger: logging.Logger):
    def decorator(fn: Callable) -> Callable:
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            logger.info(f'Starting {fn.__name__}')
            start = time.perf_counter()
            try:
                result = fn(*args, **kwargs)
                elapsed = time.perf_counter() - start
                logger.info(f'Completed {fn.__name__} in {elapsed:.3f}s')
                return result
            except Exception as e:
                logger.error(f'Failed {fn.__name__}: {type(e).__name__}: {e}')
                raise
        return wrapper
    return decorator

logger = setup_logger('ml.pipeline')

@log_step(logger)
def load_data(path: str) -> dict:
    time.sleep(0.01)  # simulate I/O
    return {'rows': 10000, 'features': 50}

@log_step(logger)
def feature_engineering(data: dict) -> dict:
    time.sleep(0.02)  # simulate compute
    return {**data, 'engineered_features': 120}

@log_step(logger)
def train_model(data: dict, config: TrainingConfig) -> dict:
    time.sleep(0.05)  # simulate training
    return {'auc': 0.924, 'model_size_mb': 48.3}

print('=== ML Pipeline Execution ===')
data = load_data('s3://bucket/train.parquet')
data = feature_engineering(data)
results = train_model(data, config)
logger.info(f'Pipeline complete: AUC={results["auc"]:.4f}')

# Structured logging for monitoring
import json
class JSONLogger:
    """Log structured JSON events (for log aggregation tools like Datadog, Splunk)."""
    def __init__(self, service: str):
        self.service = service
    
    def log(self, event: str, level: str = 'INFO', **kwargs):
        record = {
            'timestamp': datetime.utcnow().isoformat() + 'Z',
            'service': self.service,
            'level': level,
            'event': event,
            **kwargs
        }
        print(json.dumps(record))

json_logger = JSONLogger('ml-inference-api')
json_logger.log('prediction_served', model='xgboost-v2', latency_ms=12.3, prediction='positive', confidence=0.91)
json_logger.log('model_loaded', model_path='models/v2.pkl', device='cpu', load_time_ms=340)